# Variables for Drift and Diffusion Currents

| Symbol | SI Units | Description | Programming Variable |
|:-------|:----------|:-------------|:---------------------|
| $J_n$ | A·m⁻² | Electron current density | `Jn` |
| $J_p$ | A·m⁻² | Hole current density | `Jp` |
| $J$ | A·m⁻² | Total current density ($J = J_n + J_p$) | `J_total` |
| $I_n$ | A | Electron current | `In` |
| $I_p$ | A | Hole current | `Ip` |
| $A$ | m² | Cross-sectional area | `A` |
| $q$ | C | Elementary charge ($1.602\times10^{-19}$ C) | `q` |
| $n$ | m⁻³ | Electron concentration | `n_e` |
| $p$ | m⁻³ | Hole concentration | `p_h` |
| $\mu_n$ | m²·V⁻¹·s⁻¹ | Electron mobility | `mu_n` |
| $\mu_p$ | m²·V⁻¹·s⁻¹ | Hole mobility | `mu_p` |
| $\mathcal{E}$ or $E$ | V·m⁻¹ | Electric field | `E_field` |
| $v_n$ | m·s⁻¹ | Electron drift velocity | `v_n` |
| $v_p$ | m·s⁻¹ | Hole drift velocity | `v_p` |
| $D_n$ | m²·s⁻¹ | Electron diffusion coefficient | `D_n` |
| $D_p$ | m²·s⁻¹ | Hole diffusion coefficient | `D_p` |
| $\frac{dn}{dx}$ | m⁻⁴ | Electron concentration gradient | `grad_n` |
| $\frac{dp}{dx}$ | m⁻⁴ | Hole concentration gradient | `grad_p` |
| $k_B$ | J·K⁻¹ | Boltzmann constant ($1.3806\times10^{-23}$) | `k_B` |
| $T$ | K | Absolute temperature | `T` |
| $V_T = \frac{k_B T}{q}$ | V | Thermal voltage | `V_T` |
| $\sigma$ | S·m⁻¹ | Electrical conductivity | `sigma` |

---

### Derived Relationships

| Expression | Meaning |
|:------------|:---------|
| $v_n = -\mu_n \mathcal{E}$ | Electron drift velocity (opposite to field) |
| $v_p = +\mu_p \mathcal{E}$ | Hole drift velocity (along field) |
| $J_n = -q n \mu_n \mathcal{E}$ | Electron drift current density |
| $J_p = +q p \mu_p \mathcal{E}$ | Hole drift current density |
| $J_n = -q D_n \frac{dn}{dx}$ | Electron diffusion current density |
| $J_p = +q D_p \frac{dp}{dx}$ | Hole diffusion current density |
| $D = \mu \frac{k_B T}{q}$ | Einstein relation |
| $\sigma = q (n \mu_n + p \mu_p)$ | Conductivity |

---

### Notes
- In semiconductors, $n$ and $p$ vary with doping and temperature.  
- For intrinsic Si at 300 K: $n_i \approx 1.5 \times 10^{16}\ \text{m}^{-3}$  
- Mobilities are often reported in cm²/V·s; multiply by $1\times10^{-4}$ to convert to m²/V·s.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from textwrap import dedent

from enum import Enum
class MobilityModelTypes(Enum):

    

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === Constants (SI) ===
class ConstantsSI():
  q  = 1.602176634e-19
  kB = 1.380649e-23
  eps0 = 8.8541878128e-12

# ---------- Numerics ----------
def bernoulli(z):
    """Stable Bernoulli B(z) = z/(exp(z)-1) with small-z series."""
    out = np.empty_like(z, dtype=float)
    small = np.abs(z) < 1e-2
    z_s = z[small]
    out[small] = 1.0 - z_s/2.0 + (z_s*z_s)/12.0
    z_l = z[~small]
    out[~small] = z_l / (np.exp(z_l) - 1.0)
    return out

# v(E): unified drift-velocity model → μ(E) = v(E)/E
def v_sat_model(E, mu0, v_sat, beta=1.0):
    x = (mu0*E)/max(v_sat, 1e-30)
    return (mu0*E) / (1.0 + np.power(x, beta))**(1.0/beta)

def mu_field_dep(E, mu0, v_sat, beta=1.0):
    v = v_sat_model(E, mu0, v_sat, beta)
    return v / (E + 1e-30)

# ---------- Poisson ----------
def assemble_poisson_matrix(N, dx):
    main = 2.0*np.ones(N)
    off  = -1.0*np.ones(N-1)
    A = np.zeros((N,N))
    np.fill_diagonal(A, main)
    np.fill_diagonal(A[1:], off)
    np.fill_diagonal(A[:,1:], off)
    return A / (dx*dx)

def solve_poisson(phi, rho, eps, dx, phi_left, phi_right):
    """Dirichlet at both ends."""
    N = phi.size
    A = assemble_poisson_matrix(N, dx)
    b = -rho/eps
    A[0,:] = 0.0; A[0,0] = 1.0; b[0] = phi_left
    A[-1,:] = 0.0; A[-1,-1] = 1.0; b[-1] = phi_right
    phi[:] = np.linalg.solve(A, b)
    return phi

# ---------- SG continuity (∂x J = 0) ----------
def sg_continuity_solve_n(n, phi, mu_n_edges, T, dx, n_left, n_right):
    N = n.size
    Vt = kB*T/q
    dpsi = (phi[1:] - phi[:-1])/Vt
    Bp = bernoulli(+dpsi)
    Bm = bernoulli(-dpsi)
    K = q * mu_n_edges * Vt / dx
    a = -K * Bm                 # sub
    c = -K * Bp                 # super
    b = np.zeros(N); b[1:-1] = -(a[1:] + c[:-1])
    A = np.zeros((N,N))
    for i in range(1, N-1):
        A[i,i] = b[i]
        A[i,i-1] = a[i]
        A[i,i+1] = c[i-1]
    A[0,0] = 1.0; A[-1,-1] = 1.0
    rhs = np.zeros(N); rhs[0] = n_left; rhs[-1] = n_right
    n[:] = np.linalg.solve(A, rhs)
    return n

def sg_continuity_solve_p(p, phi, mu_p_edges, T, dx, p_left, p_right):
    N = p.size
    Vt = kB*T/q
    dpsi = (phi[1:] - phi[:-1])/Vt
    Bp = bernoulli(+dpsi)
    Bm = bernoulli(-dpsi)
    K = q * mu_p_edges * Vt / dx
    a = -K * Bp                 # sub (note swap vs electrons)
    c = -K * Bm                 # super
    b = np.zeros(N); b[1:-1] = -(a[1:] + c[:-1])
    A = np.zeros((N,N))
    for i in range(1, N-1):
        A[i,i] = b[i]
        A[i,i-1] = a[i]
        A[i,i+1] = c[i-1]
    A[0,0] = 1.0; A[-1,-1] = 1.0
    rhs = np.zeros(N); rhs[0] = p_left; rhs[-1] = p_right
    p[:] = np.linalg.solve(A, rhs)
    return p

def compute_edge_mobility(mu_cell, E_cell, model="constant", v_sat=1e5, beta=1.0):
    mu_edges = 0.5*(mu_cell[:-1] + mu_cell[1:])
    if model == "constant":
        return mu_edges
    E_edges = 0.5*(E_cell[:-1] + E_cell[1:])
    return mu_field_dep(E_edges, mu_edges, v_sat, beta)

def currents(phi, n, p, mu_n_edges, mu_p_edges, T, dx):
    Vt = kB*T/q
    dpsi = (phi[1:] - phi[:-1])/Vt
    Bp = bernoulli(+dpsi); Bm = bernoulli(-dpsi)
    Jn = (q * mu_n_edges * Vt / dx) * ( n[:-1]*Bp - n[1:]*Bm )
    Jp = -(q * mu_p_edges * Vt / dx) * ( p[:-1]*Bm - p[1:]*Bp )
    return Jn, Jp

# ---------- Driver ----------
def run_iv(
    L=1e-6, Nx=121, T=300.0,
    ND=1e22, NA=0.0, ni=1.45e16,
    eps_rel=11.7,
    mu_n0=0.135, mu_p0=0.048,       # 1350, 480 cm^2/(V s)
    v_sat_n=1e5, v_sat_p=8e4,       # m/s
    mobility_model="constant",      # "constant" or "field"
    V_list=np.linspace(0.0, 0.5, 7),
    gummel_maxit=25, tol=1e-6
):
    dx = L/(Nx-1)
    x = np.linspace(0.0, L, Nx)
    eps = eps_rel*eps0

    # Init: n-type bar, majority carriers set by ND, minority by ni^2 / n
    phi = np.zeros(Nx)
    n = np.maximum(ND-NA, 1.0)*np.ones(Nx)
    p = (ni*ni/np.maximum(n, 1.0))*np.ones(Nx)

    mu_n_cell = mu_n0*np.ones(Nx)
    mu_p_cell = mu_p0*np.ones(Nx)

    I_list = []
    for Vapp in V_list:
        phi_left, phi_right = 0.0, Vapp
        # Ohmic contact values (pedagogical, keeps problem well-posed)
        n_left = max(ND, ni); p_left = ni*ni/n_left
        n_right = max(ND, ni); p_right = ni*ni/n_right

        for _ in range(gummel_maxit):
            # Poisson
            rho = q*(p - n + ND - NA)
            phi_old = phi.copy()
            phi = solve_poisson(phi, rho, eps, dx, phi_left, phi_right)

            # Edge mobilities
            E_cell = -(np.gradient(phi, dx))
            mu_n_edges = compute_edge_mobility(mu_n_cell, E_cell,
                                               model=("field" if mobility_model=="field" else "constant"),
                                               v_sat=v_sat_n, beta=1.0)
            mu_p_edges = compute_edge_mobility(mu_p_cell, E_cell,
                                               model=("field" if mobility_model=="field" else "constant"),
                                               v_sat=v_sat_p, beta=1.0)
            # Continuity (no recombination in this minimal DC example)
            n_old = n.copy()
            n = sg_continuity_solve_n(n, phi, mu_n_edges, T, dx, n_left, n_right)
            p_old = p.copy()
            p = sg_continuity_solve_p(p, phi, mu_p_edges, T, dx, p_left, p_right)

            # Convergence check (∞-norm)
            err = max(np.max(np.abs(phi-phi_old)), np.max(np.abs(n-n_old)), np.max(np.abs(p-p_old)))
            if err < tol:
                break

        Jn, Jp = currents(phi, n, p, mu_n_edges, mu_p_edges, T, dx)
        I_list.append(np.mean(Jn + Jp))  # A/m^2

    return x, phi, n, p, np.array(V_list), np.array(I_list)

def demo():
    V_list = np.linspace(0.0, 0.6, 7)
    _, _, _, _, V, I_const = run_iv(V_list=V_list, mobility_model="constant")
    _, _, _, _, _, I_field = run_iv(V_list=V_list, mobility_model="field")
    plt.figure()
    plt.plot(V, I_const, marker='o', label='constant mobility')
    plt.plot(V, I_field, marker='s', label='field-dependent mobility')
    plt.xlabel('Voltage (V)')
    plt.ylabel('Current density (A/m$^2$)')
    plt.title('1D Drift–Diffusion IV (Scharfetter–Gummel)')
    plt.legend()
    plt.grid(True, which='both', ls=':')
    plt.tight_layout()
    plt.savefig('iv_curve.png', dpi=160)

if __name__ == "__main__":
    demo()
